<a href="https://colab.research.google.com/github/Chakrika-Reddy/2520090044_GenAi/blob/main/Skill_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [2]:
!pip install tritonclient[all] fastapi uvicorn pydantic pillow torchvision pyngrok nest_asyncio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 616.8/616.8 kB 8.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 20.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 26.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 16.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 20.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 26.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 23.6 MB/s  0:00:00
  Attempting uninstall: ml-dtypes
    Found existing installation: ml_dtypes 0.6.0
    Uninstalling ml_dtypes-0.6.0:
      Successfully uninstalled ml_dtypes-0.6.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14/14 [cuda-python]


In [4]:
import os

os.makedirs("model_repository/vision_model/1", exist_ok=True)

In [10]:
import os
import sys

target_dir = "model_repository/vision_model/1"

print(f"Current Working Directory: {os.getcwd()}")
print(f"Target directory to create: {os.path.abspath(target_dir)}")

# Check if target already exists as a file
if os.path.exists(target_dir):
    if os.path.isdir(target_dir):
        print(f"SUCCESS: '{target_dir}' already exists as a directory.")
    else:
        print(f"ERROR: '{target_dir}' exists but is a FILE, which will prevent directory creation!")
else:
    try:
        os.makedirs(target_dir, exist_ok=True)
        print(f"SUCCESS: Successfully created directory path: '{target_dir}'")
    except Exception as e:
        print(f"ERROR: Failed to create directories due to: {e}")

# Test write permissions in the created path
try:
    test_file_path = os.path.join(target_dir, ".write_test")
    with open(test_file_path, "w") as f:
        f.write("test")
    os.remove(test_file_path)
    print("SUCCESS: Write permissions verified successfully in the target directory.")
except Exception as e:
    print(f"ERROR: Target directory exists but is NOT writeable: {e}")

Current Working Directory: /content
Target directory to create: /content/model_repository/vision_model/1
SUCCESS: 'model_repository/vision_model/1' already exists as a directory.
SUCCESS: Write permissions verified successfully in the target directory.


In [5]:
config_content = """
name: "vision_model"
platform: "onnxruntime_onnx"
max_batch_size: 8
input [
  {
    name: "input"
    data_type: TYPE_FP32
    dims: [ 3, 224, 224 ]
  }
]
output [
  {
    name: "output"
    data_type: TYPE_FP32
    dims: [ 1000 ]
  }
]
"""

with open("model_repository/vision_model/config.pbtxt", "w") as f:
    f.write(config_content)

In [7]:
!pip install onnx onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 43.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 754.2/754.2 kB 31.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [onnxscript]


In [8]:
import torch
import torchvision.models as models

model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT).eval()
dummy_input = torch.randn(1, 3, 224, 224)

torch.onnx.export(
    model,
    dummy_input,
    "model_repository/vision_model/1/model.onnx",
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
)

/tmp/ipykernel_2937/2592845645.py:7: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(


[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅


ONNXProgram(
    model=
        <
            ir_version=10,
            opset_imports={'': 20},
            producer_name='pytorch',
            producer_version='2.14.1+cpu',
            domain=None,
            model_version=None,
        >
        graph(
            name=main_graph,
            inputs=(
                %"input"<FLOAT,[batch_size,3,224,224]>
            ),
            outputs=(
                %"output"<FLOAT,[batch_size,1000]>
            ),
            initializers=(
                %"conv1.weight"<FLOAT,[64,3,7,7]>{Tensor(...)},
                %"layer1.0.conv1.weight"<FLOAT,[64,64,1,1]>{Tensor(...)},
                %"layer1.0.conv2.weight"<FLOAT,[64,64,3,3]>{Tensor(...)},
                %"layer1.0.conv3.weight"<FLOAT,[256,64,1,1]>{Tensor(...)},
                %"layer1.0.downsample.0.weight"<FLOAT,[256,64,1,1]>{Tensor(...)},
                %"layer1.1.conv1.weight"<FLOAT,[64,256,1,1]>{Tensor(...)},
                %"layer1.1.conv2.weight"<FLOAT,[64,64,3,3]>{Te

In [13]:
!pip install onnxruntime

import onnxruntime as ort
import numpy as np

# Path to the exported model
model_path = "model_repository/vision_model/1/model.onnx"

# Create an ONNX Runtime inference session
session = ort.InferenceSession(model_path)

# Generate a dummy input matching the expected dimensions [batch_size, 3, 224, 224]
dummy_input = np.random.randn(1, 3, 224, 224).astype(np.float32)

# Get inputs and run the model
input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name

results = session.run([output_name], {input_name: dummy_input})

print("Model inference ran successfully!")
print("Output shape:", results[0].shape)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 67.5 MB/s  0:00:00
Model inference ran successfully!
Output shape: (1, 1000)


In [14]:
app_code = """
from fastapi import FastAPI, UploadFile, File
import tritonclient.http as httpclient
import numpy as np
from PIL import Image
import io

app = FastAPI(title="VisionPort API")
triton_client = httpclient.InferenceServerClient(url="localhost:8000")

@app.get("/health")
def health_check():
    return {"status": "running", "triton_live": triton_client.is_server_live()}

@app.post("/predict")
async def predict(file: UploadFile = File(...)):
    image_data = await file.read()
    image = Image.open(io.BytesIO(image_data)).convert("RGB")
    image = image.resize((224, 224))

    img_array = np.array(image, dtype=np.float32) / 255.0
    img_array = np.transpose(img_array, (2, 0, 1))
    img_array = np.expand_dims(img_array, axis=0)

    inputs = [httpclient.InferInput("input", img_array.shape, "FP32")]
    inputs[0].set_data_from_numpy(img_array)
    outputs = [httpclient.InferRequestedOutput("output")]

    response = triton_client.infer(model_name="vision_model", inputs=inputs, outputs=outputs)
    result = response.as_numpy("output")

    return {
        "prediction_shape": list(result.shape),
        "top_class_index": int(np.argmax(result))
    }
"""

with open("app.py", "w") as f:
    f.write(app_code)

In [16]:
import nest_asyncio
import uvicorn
import threading
import socket
from pyngrok import ngrok

# 1. TODO: Replace with your actual ngrok authtoken from https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_TOKEN = "YOUR_NGROK_AUTHTOKEN_HERE"
if NGROK_TOKEN != "YOUR_NGROK_AUTHTOKEN_HERE":
    ngrok.set_auth_token(NGROK_TOKEN)
else:
    print("⚠️ Please set your NGROK_TOKEN in the code to establish a public URL connection.")

nest_asyncio.apply()

# Helper to find a free port dynamically if 8080 is blocked
def find_free_port():
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.bind(('localhost', 0))
    port = s.getsockname()[1]
    s.close()
    return port

port = 8080
try:
    # Test if 8080 is free
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.bind(('0.0.0.0', port))
    s.close()
except OSError:
    port = find_free_port()
    print(f"Port 8080 was busy, using fallback port: {port}")

def run_fastapi():
    uvicorn.run("app:app", host="0.0.0.0", port=port)

threading.Thread(target=run_fastapi, daemon=True).start()

if NGROK_TOKEN != "YOUR_NGROK_AUTHTOKEN_HERE":
    try:
        public_url = ngrok.connect(port)
        print("Public FastAPI URL:", public_url)
    except Exception as e:
        print("Failed to start ngrok tunnel:", e)
else:
    print(f"FastAPI is running locally inside Colab on port {port}. (Requires ngrok authtoken to tunnel externally)")

⚠️ Please set your NGROK_TOKEN in the code to establish a public URL connection.
Port 8080 was busy, using fallback port: 54791
FastAPI is running locally inside Colab on port 54791. (Requires ngrok authtoken to tunnel externally)


INFO:     Started server process [2937]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
